In [1]:
# Cấu hình số luồng CPU trước khi import NumPy/Pandas/TensorFlow.
import os

logical_cpus = os.cpu_count() or 1
compute_threads = max(1, logical_cpus // 2)
interop_threads = min(8, max(1, logical_cpus // 4))

os.environ['OPENBLAS_NUM_THREADS'] = str(compute_threads)
os.environ['MKL_NUM_THREADS'] = str(compute_threads)
os.environ['OMP_NUM_THREADS'] = str(compute_threads)
os.environ['NUMEXPR_NUM_THREADS'] = str(compute_threads)
os.environ['TF_NUM_INTRAOP_THREADS'] = str(compute_threads)
os.environ['TF_NUM_INTEROP_THREADS'] = str(interop_threads)

# Chỉ bỏ comment nếu cần tắt oneDNN để kiểm tra sai khác số học.
# os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print(f'CPU logic: {logical_cpus}')
print(f'BLAS/TF intra-op: {compute_threads} luồng')
print(f'TF inter-op: {interop_threads} luồng')


CPU logic: 88
BLAS/TF intra-op: 44 luồng
TF inter-op: 8 luồng


In [2]:
%%time

# =========================================================
# BƯỚC 1: Import các thư viện cần thiết
# =========================================================

import pandas as pd
import glob
import os
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# TensorFlow / Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau,
    EarlyStopping,
    ModelCheckpoint
)
from tensorflow.keras.utils import to_categorical

2026-08-05 22:04:00.946976: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-05 22:04:01.004611: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


CPU times: user 6.42 s, sys: 3.52 s, total: 9.94 s
Wall time: 2.57 s


2026-08-05 22:04:02.109506: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
%%time

# =========================================================
# BƯỚC 2: ĐỌC VÀ GỘP DỮ LIỆU (LOAD & COMBINE DATA)
# =========================================================

# Notebook và thư mục 'dataset' cùng nằm trong thư mục AI_Security.
# resolve() giúp in ra đường dẫn tuyệt đối để dễ kiểm tra khi có lỗi.
path = os.path.abspath('dataset')

# Tìm tất cả các file có đuôi .csv trong thư mục đó
all_files = sorted(glob.glob(os.path.join(path, '*.csv')))

# Tạo một danh sách rỗng để chứa dữ liệu của từng file
li = []

# Kiểm tra nếu tìm thấy file CSV thì mới tiến hành đọc
if all_files:
    print(f"🎉 Tìm thấy {len(all_files)} file CSV trong thư mục '{path}'. Bắt đầu đọc...")
    print("-------------------------------------------------")
    
    for filename in all_files:
        print(f"--> Đang đọc file: {filename}")
        # Đọc file CSV bằng pandas
        df = pd.read_csv(filename, index_col=None, header=0, low_memory=False)
        li.append(df)

    # Gộp tất cả các bảng dữ liệu lẻ lại thành 1 bảng duy nhất theo hàng (axis=0)
    dataset = pd.concat(li, axis=0, ignore_index=True)

    # Tạo thư mục đầu ra và lưu dữ liệu sau khi gộp.
    output_dir = os.path.abspath('output_after_preprocess')
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, 'dataset_merged.csv')
    dataset.to_csv(output_file, index=False, encoding='utf-8', chunksize=100_000)
    
    print("-------------------------------------------------")
    print("✅ HOÀN TẤT ĐỌC VÀ GỘP DỮ LIỆU!")
    print(f"📊 Tổng số dòng dữ liệu (Rows): {dataset.shape[0]:,}")
    print(f"📊 Tổng số cột đặc trưng (Columns): {dataset.shape[1]}")
    print(f"💾 Đã lưu dữ liệu sau khi gộp tại: {output_file}")
    print("-------------------------------------------------\n")
    
    # Hiển thị 5 dòng đầu tiên của bảng dữ liệu gộp để kiểm tra cấu trúc
    display(dataset.head())

else:
    raise FileNotFoundError(
        f"Không tìm thấy file CSV trong: {path}. "
        f"Thư mục làm việc hiện tại là: {os.getcwd()}"
    )


🎉 Tìm thấy 4 file CSV trong thư mục '/home/ubuntu/sepcung/AI_Security/dataset'. Bắt đầu đọc...
-------------------------------------------------
--> Đang đọc file: /home/ubuntu/sepcung/AI_Security/dataset/DDoS1-Tuesday-20-02-2018.csv
--> Đang đọc file: /home/ubuntu/sepcung/AI_Security/dataset/DDoS2-Wednesday-21-02-2018.csv
--> Đang đọc file: /home/ubuntu/sepcung/AI_Security/dataset/DoS1-Thursday-15-02-2018.csv
--> Đang đọc file: /home/ubuntu/sepcung/AI_Security/dataset/DoS2-Friday-16-02-2018.csv
-------------------------------------------------
✅ HOÀN TẤT ĐỌC VÀ GỘP DỮ LIỆU!
📊 Tổng số dòng dữ liệu (Rows): 4,194,299
📊 Tổng số cột đặc trưng (Columns): 80
💾 Đã lưu dữ liệu sau khi gộp tại: /home/ubuntu/sepcung/AI_Security/output_after_preprocess/dataset_merged.csv
-------------------------------------------------



,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,6,20/02/2018 08:34:07,888751,11,11,1249,1969,736,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
1,0,0,20/02/2018 08:33:22,112642816,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56300000.0,7.071068,56300000,56300000,Benign
2,0,0,20/02/2018 08:36:11,112642712,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56300000.0,18.384776,56300000,56300000,Benign
3,0,0,20/02/2018 08:39:00,112642648,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56300000.0,5.656854,56300000,56300000,Benign
4,0,0,20/02/2018 08:41:49,112642702,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56300000.0,65.053824,56300000,56300000,Benign


CPU times: user 1min 46s, sys: 16.4 s, total: 2min 3s
Wall time: 2min 3s


In [4]:
%%time

# =========================================================
# BƯỚC 3: XỬ LÝ GIÁ TRỊ THIẾU, VÔ CỰC VÀ BẢN GHI TRÙNG LẶP
# =========================================================

# Thay các giá trị vô cực dương/âm thành NaN để có thể xóa ở bước dropna.
dataset.replace([np.inf, -np.inf], np.nan, inplace=True)

# Chuẩn hóa các cột kiểu object, giúp tương thích tốt hơn với các phiên bản pandas mới.
dataset = dataset.infer_objects(copy=False)

# Đếm số dòng và số giá trị lỗi trước khi làm sạch dữ liệu.
rows_before = dataset.shape[0]
missing_before = dataset.isna().sum().sum()

# Xóa tất cả các dòng còn chứa giá trị thiếu.
dataset.dropna(inplace=True)
dataset.reset_index(drop=True, inplace=True)

rows_after_dropna = dataset.shape[0]

# ---------------------------------------------------------
# Loại bỏ bản ghi trùng lặp
# ---------------------------------------------------------
# CSE-CIC-IDS2018 chứa rất nhiều flow trùng khít nhau, đặc biệt trong
# các đợt flood (HOIC/LOIC sinh ra hàng trăm nghìn flow giống hệt).
# Nếu không loại bỏ, các dòng giống nhau sẽ rơi vào cả tập train lẫn
# tập test khi chia dữ liệu, khiến mô hình chỉ cần ghi nhớ là đạt
# độ chính xác gần như tuyệt đối (rò rỉ dữ liệu).
#
# So trùng trên toàn bộ đặc trưng + Label, KHÔNG tính Timestamp:
# hai flow giống hệt nhau nhưng lệch nhau vài giây vẫn là trùng lặp.

duplicate_subset = [
    column
    for column in dataset.columns
    if column != 'Timestamp'
]

duplicate_mask = dataset.duplicated(subset=duplicate_subset, keep='first')
duplicate_count = int(duplicate_mask.sum())

if duplicate_count > 0:
    dataset = dataset.loc[~duplicate_mask].copy()
    dataset.reset_index(drop=True, inplace=True)

rows_after = dataset.shape[0]

# Kiểm tra mâu thuẫn nhãn: cùng một vector đặc trưng nhưng gán 2 nhãn khác nhau.
# Đây là nhiễu nhãn, không thể học được, chỉ cần biết quy mô để báo cáo.
feature_only_subset = [
    column
    for column in dataset.columns
    if column not in ('Timestamp', 'Label')
]

conflicting_count = int(
    dataset.duplicated(subset=feature_only_subset, keep=False).sum()
)

# Lưu dữ liệu sau khi xử lý NaN/vô cực/trùng lặp vào thư mục đầu ra đã có.
output_after_dir = os.path.abspath('output_after_preprocess')
os.makedirs(output_after_dir, exist_ok=True)
step3_output_file = os.path.join(
    output_after_dir,
    'dataset_after_missing_infinite.csv'
)
dataset.to_csv(
    step3_output_file,
    index=False,
    encoding='utf-8',
    chunksize=100_000
)

print(f"Số giá trị thiếu/vô cực được tìm thấy: {missing_before:,}")
print(f"Số dòng đã xóa vì thiếu/vô cực: {rows_before - rows_after_dropna:,}")
print(f"Số dòng trùng lặp đã xóa: {duplicate_count:,}")
print(
    f"Số dòng có đặc trưng trùng nhau nhưng khác nhãn (nhiễu nhãn còn lại): "
    f"{conflicting_count:,}"
)
print(f"Tổng số dòng đã loại: {rows_before - rows_after:,}")
print("Kích thước dữ liệu sau Bước 3:", dataset.shape)
print(f"Đã lưu dữ liệu sau Bước 3 tại: {step3_output_file}")

display(dataset.head())

Số giá trị thiếu/vô cực được tìm thấy: 21,804
Số dòng đã xóa vì thiếu/vô cực: 10,902
Số dòng trùng lặp đã xóa: 1,169,515
Số dòng có đặc trưng trùng nhau nhưng khác nhãn (nhiễu nhãn còn lại): 0
Tổng số dòng đã loại: 1,180,417
Kích thước dữ liệu sau Bước 3: (3013882, 80)
Đã lưu dữ liệu sau Bước 3 tại: /home/ubuntu/sepcung/AI_Security/output_after_preprocess/dataset_after_missing_infinite.csv


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,6,20/02/2018 08:34:07,888751,11,11,1249,1969,736,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
1,0,0,20/02/2018 08:33:22,112642816,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56300000.0,7.071068,56300000,56300000,Benign
2,0,0,20/02/2018 08:36:11,112642712,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56300000.0,18.384776,56300000,56300000,Benign
3,0,0,20/02/2018 08:39:00,112642648,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56300000.0,5.656854,56300000,56300000,Benign
4,0,0,20/02/2018 08:41:49,112642702,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56300000.0,65.053824,56300000,56300000,Benign


CPU times: user 1min 17s, sys: 10.2 s, total: 1min 27s
Wall time: 1min 27s


In [5]:
%%time

# =========================================================
# BƯỚC 4: LỌC DỮ LIỆU CHO BÀI TOÁN PHÁT HIỆN DDoS
# =========================================================

# Chuẩn hóa tên nhãn để tránh sai lệch do khoảng trắng thừa.
dataset['Label'] = dataset['Label'].astype(str).str.strip()

print("Số lượng từng nhãn trước khi lọc:")
display(dataset['Label'].value_counts())

# Chỉ giữ lưu lượng bình thường và các biến thể DDoS.
# Giữ Benign làm lớp đối chứng để huấn luyện mô hình phân loại nhị phân.
ddos_mapping = {
    'Benign': 'Benign',
    'DDOS attack-HOIC': 'DDoS',
    'DDoS attacks-LOIC-HTTP': 'DDoS',
    'DDOS attack-LOIC-UDP': 'DDoS',
    'DDoS': 'DDoS'  # Cho phép chạy lại cell mà không làm mất dữ liệu.
}

rows_before = dataset.shape[0]
labels_removed = sorted(set(dataset['Label'].unique()) - set(ddos_mapping))

# Loại toàn bộ kiểu tấn công khác, sau đó gộp các biến thể DDoS.
dataset = dataset[dataset['Label'].isin(ddos_mapping)].copy()

# Giữ lại tên tấn công gốc trước khi gộp về nhị phân.
# Cột này KHÔNG dùng làm đặc trưng huấn luyện, chỉ dùng ở bước đánh giá
# để báo cáo recall riêng cho từng biến thể (HOIC / LOIC-HTTP / LOIC-UDP).
# Nếu chạy lại cell, cột cũ đã có thì giữ nguyên để không mất thông tin.
if 'Attack Type' not in dataset.columns:
    dataset['Attack Type'] = dataset['Label']

dataset['Label'] = dataset['Label'].map(ddos_mapping)
dataset.reset_index(drop=True, inplace=True)

rows_after = dataset.shape[0]

if dataset.empty:
    raise ValueError("Không tìm thấy dữ liệu Benign hoặc DDoS sau khi lọc.")

# Lưu dữ liệu sau khi lọc và gán nhãn vào thư mục đầu ra.
output_after_dir = os.path.abspath('output_after_preprocess')
os.makedirs(output_after_dir, exist_ok=True)
step4_output_file = os.path.join(
    output_after_dir,
    'dataset_after_labeling.csv'
)
dataset.to_csv(
    step4_output_file,
    index=False,
    encoding='utf-8',
    chunksize=100_000
)

print("Các nhãn đã loại bỏ:", labels_removed or "Không có")
print(f"Số dòng đã loại bỏ: {rows_before - rows_after:,}")
print("Số lượng mẫu cho bài toán DDoS nhị phân:")
display(dataset['Label'].value_counts())

print("Số lượng theo từng biến thể tấn công (dùng cho báo cáo đánh giá):")
display(dataset['Attack Type'].value_counts())

print("Kích thước dữ liệu sau Bước 4:", dataset.shape)
print("Danh sách nhãn cuối cùng:", sorted(dataset['Label'].unique()))
print(f"Đã lưu dữ liệu sau Bước 4 tại: {step4_output_file}")

Số lượng từng nhãn trước khi lọc:


Label
Benign                      2041359
DDoS attacks-LOIC-HTTP       575364
DDOS attack-HOIC             198861
DoS attacks-Hulk             145199
DoS attacks-GoldenEye         41406
DoS attacks-Slowloris          9908
DDOS attack-LOIC-UDP           1730
DoS attacks-SlowHTTPTest         55
Name: count, dtype: int64

Các nhãn đã loại bỏ: ['DoS attacks-GoldenEye', 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest', 'DoS attacks-Slowloris']
Số dòng đã loại bỏ: 196,568
Số lượng mẫu cho bài toán DDoS nhị phân:


Label
Benign    2041359
DDoS       775955
Name: count, dtype: int64

Số lượng theo từng biến thể tấn công (dùng cho báo cáo đánh giá):


Attack Type
Benign                    2041359
DDoS attacks-LOIC-HTTP     575364
DDOS attack-HOIC           198861
DDOS attack-LOIC-UDP         1730
Name: count, dtype: int64

Kích thước dữ liệu sau Bước 4: (2817314, 81)
Danh sách nhãn cuối cùng: ['Benign', 'DDoS']
Đã lưu dữ liệu sau Bước 4 tại: /home/ubuntu/sepcung/AI_Security/output_after_preprocess/dataset_after_labeling.csv
CPU times: user 54.9 s, sys: 4.38 s, total: 59.3 s
Wall time: 59.3 s


In [6]:
%%time

# =========================================================
# BƯỚC 5: ĐỌC DỮ LIỆU BƯỚC 4 VÀ MÃ HÓA NHÃN LABEL
# =========================================================

import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Có thể chạy Bước 5 từ file đầu ra của Bước 4, không phụ thuộc
# vào biến dataset còn tồn tại trong RAM từ các bước trước.
step4_input_file = os.path.abspath(os.path.join(
    'output_after_preprocess',
    'dataset_after_labeling.csv'
))

if not os.path.isfile(step4_input_file):
    raise FileNotFoundError(
        f'Không tìm thấy dữ liệu đầu vào của Bước 5: {step4_input_file}'
    )

print(f'Đang đọc dữ liệu từ: {step4_input_file}')
dataset = pd.read_csv(step4_input_file, low_memory=False)
print(f'Đã đọc {dataset.shape[0]:,} dòng và {dataset.shape[1]} cột.')

if 'Label' not in dataset.columns:
    raise KeyError("Dữ liệu đầu vào không có cột 'Label'.")

# ---------------------------------------------------------
# Chuẩn hóa Timestamp về Unix time
# ---------------------------------------------------------
# QUAN TRỌNG: Timestamp KHÔNG được dùng làm đặc trưng huấn luyện.
# Trong CSE-CIC-IDS2018, các đợt tấn công diễn ra trong những khung
# giờ cố định và hai ngày Thứ Năm 15/02, Thứ Sáu 16/02 chỉ còn lại
# lưu lượng Benign sau khi lọc ở Bước 4. Nếu đưa Timestamp vào X,
# mô hình chỉ cần học "lịch tấn công" là đạt gần 100% mà không hề
# học đặc trưng mạng nào - kết quả sẽ vô dụng khi triển khai thật.
#
# Cột này chỉ được giữ lại để chia dữ liệu train/test theo thời gian
# ở Bước 7 của notebook DNN, sau đó bị loại khỏi X.
if 'Timestamp' in dataset.columns:
    if not pd.api.types.is_numeric_dtype(dataset['Timestamp']):
        timestamp_parsed = pd.to_datetime(
            dataset['Timestamp'],
            errors='coerce',
            dayfirst=True
        )
        invalid_timestamp_count = timestamp_parsed.isna().sum()

        if invalid_timestamp_count > 0:
            print(f'Loại {invalid_timestamp_count:,} dòng vì Timestamp không hợp lệ.')
            valid_timestamp_mask = timestamp_parsed.notna()
            dataset = dataset.loc[valid_timestamp_mask].copy()
            timestamp_parsed = timestamp_parsed.loc[valid_timestamp_mask]

        dataset['Timestamp'] = (
            timestamp_parsed.astype('int64') // 10**9
        ).astype('int64')
        dataset.reset_index(drop=True, inplace=True)
        print('Đã mã hóa Timestamp thành Unix time (chỉ dùng để chia dữ liệu).')
    else:
        print('Timestamp đã là dạng số (chỉ dùng để chia dữ liệu).')

    # Thống kê số mẫu theo từng ngày để kiểm tra kế hoạch chia theo thời gian.
    day_series = pd.to_datetime(
        dataset['Timestamp'],
        unit='s'
    ).dt.strftime('%Y-%m-%d')

    print('\nPhân bố nhãn theo từng ngày:')
    display(pd.crosstab(day_series, dataset['Label']))

# Chuẩn hóa và kiểm tra nhãn trước khi mã hóa.
dataset['Label'] = dataset['Label'].astype(str).str.strip()
labels_before_encoding = sorted(dataset['Label'].unique())
if len(labels_before_encoding) < 2:
    raise ValueError(
        f'Cần ít nhất 2 lớp để huấn luyện, hiện chỉ có: {labels_before_encoding}'
    )

# Mã hóa cột Label từ dạng chữ sang dạng số.
label_encoder = LabelEncoder()
dataset['Label'] = label_encoder.fit_transform(dataset['Label'])

# Lưu bảng ánh xạ để biết số nào tương ứng với nhãn nào.
label_mapping = dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
))

print('Bảng mã hóa nhãn:')
for label, number in label_mapping.items():
    print(f'{label} -> {number}')

print('\nKích thước dữ liệu sau Bước 5:', dataset.shape)
display(dataset.head())

Đang đọc dữ liệu từ: /home/ubuntu/sepcung/AI_Security/output_after_preprocess/dataset_after_labeling.csv
Đã đọc 2,817,314 dòng và 81 cột.
Đã mã hóa Timestamp thành Unix time (chỉ dùng để chia dữ liệu).

Phân bố nhãn theo từng ngày:


Label,Benign,DDoS
Timestamp,,
2018-02-15,823148,0
2018-02-16,446645,0
2018-02-20,410761,575364
2018-02-21,360805,200591


Bảng mã hóa nhãn:
Benign -> 0
DDoS -> 1

Kích thước dữ liệu sau Bước 5: (2817314, 81)


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Attack Type
0,22,6,1519115647,888751,11,11,1249,1969,736,0,...,0.0,0.0,0,0,0.0,0.000000,0,0,0,Benign
1,0,0,1519115602,112642816,3,0,0,0,0,0,...,0.0,0.0,0,0,56300000.0,7.071068,56300000,56300000,0,Benign
2,0,0,1519115771,112642712,3,0,0,0,0,0,...,0.0,0.0,0,0,56300000.0,18.384776,56300000,56300000,0,Benign
3,0,0,1519115940,112642648,3,0,0,0,0,0,...,0.0,0.0,0,0,56300000.0,5.656854,56300000,56300000,0,Benign
4,0,0,1519116109,112642702,3,0,0,0,0,0,...,0.0,0.0,0,0,56300000.0,65.053824,56300000,56300000,0,Benign


CPU times: user 21.6 s, sys: 2.98 s, total: 24.6 s
Wall time: 24.6 s
